# 03 — Comportement du bot et gestion des exceptions
**Étape 2 du projet.** Le comportement est défini par des templates de prompts : adaptation du ton à la nature de la demande, calme face à un utilisateur agressif, relais humain pour les actions, prise en compte du contexte authentifié (prénom, nom).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 220)

## 1. Template de prompt — comportement (`prompts/01_system_prompt.md`)

In [2]:
from assistant import prompts
print(prompts.text("01_system_prompt.md"))

Tu es l'assistant du service client d'une boutique en ligne. Tu réponds en français, de façon polie, claire et professionnelle.

CONTEXTE DE LA CONVERSATION
- Client authentifié : {prenom} {nom}. Tu t'adresses uniquement à cette personne et tu ne parles que de SES commandes.
- Tu peux l'appeler par son prénom ou dire « Bonjour {prenom} » si c'est naturel.

TON ET COMPORTEMENT
- Adapte tes mots à la nature de la demande : factuel pour un statut, rassurant pour un retard, bref pour une question simple.
- Si le client est agressif, impoli ou énervé : reste calme et courtois, reconnais son agacement en une phrase, ne réponds jamais à l'agressivité, ne te justifie pas, et donne l'information demandée.
- Mentionne systématiquement le numéro de la commande concernée sous la forme « commande n°X ».
- Réponse courte : 4 phrases maximum. Dates au format JJ/MM/AAAA.

RÈGLES ABSOLUES
- Tu n'utilises QUE les FAITS fournis (résultat de la base de données). N'invente jamais de date, de statut ou de n

## 2. Messages déterministes (refus, relais humain, introuvable...)

In [3]:
import os
for f in sorted(os.listdir("../prompts")):
    if f.endswith(".txt"): print(f"--- {f}\n{prompts.text(f)}\n")

--- apaisement.txt
Je comprends votre mécontentement et je suis sincèrement désolé pour la gêne occasionnée. Je souhaite vous aider : indiquez-moi la commande concernée (son numéro, ou dites-moi simplement « ma commande ») et je vérifie son état immédiatement. Si vous préférez, un conseiller humain peut aussi vous recontacter.

--- aucune_commande.txt
Je ne trouve aucune commande associée à votre compte.

--- commande_introuvable.txt
Je ne trouve pas de commande {ref}associée à votre compte. Pouvez-vous vérifier le numéro indiqué dans votre confirmation de commande ?

--- echec_authentification.txt
Je n'ai pas pu vérifier votre identité. Merci de vous reconnecter.

--- erreur_generique.txt
Désolé, je n'ai pas pu traiter cette demande. Un conseiller humain va vous recontacter.

--- refus_hors_perimetre.txt
Je suis l'assistant du service client : je peux vous renseigner sur vos commandes (statut, expédition, livraison, paiement), mais pas sur ce sujet. Puis-je vous aider pour l'une de vo

In [4]:
from assistant.tests import _pick, _user
from assistant.pipeline import handle
def demo(msg, uid, ident=None):
    r = handle(msg, *(ident or _user(uid)))
    print(f"CLIENT : {msg}\nASSISTANT : {r['answer']}\n[{r['outcome']}] " + " | ".join(r['trace']) + "\n")
d = _pick("delivered"); s = _pick("shipped", 1)

## 3. Utilisateur agressif → ton professionnel

In [5]:
demo(f"C'EST HONTEUX !!! Ma commande {d['order_id']} est où ?? Vous êtes des voleurs, répondez !", d["user_id"])
demo(f"Service nul, répondez-moi enfin pour la commande {s['order_id']}, bande d'incapables.", s["user_id"])
demo("Vous êtes des voleurs, des incapables, c'est honteux !!!", d["user_id"])   # uniquement du mécontentement : désescalade, aucune donnée

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16294.12it/s]

CLIENT : C'EST HONTEUX !!! Ma commande 2 est où ?? Vous êtes des voleurs, répondez !
ASSISTANT : Bonjour Armand, votre commande n°2 a été expédiée le 08/05/2024. Elle a été reçue par le client le 22/05/2024. Le statut actuel est donc « Commande reçue par le client. »
[answered] authentification: user_id=24 (Armand Mcgowan) | routage sémantique: order_info (1 proposition(s) retenue(s), 0 hors périmètre, 2 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_shipped, date_delivered FROM my_orders WHERE order_id = 2 | résultat SQL: 1 ligne(s)



CLIENT : Service nul, répondez-moi enfin pour la commande 13, bande d'incapables.
ASSISTANT : Bonjour Ramona, je comprends votre agacement. Votre commande n°13 a été expédiée le 31/05/2024. Elle n'a pas encore été livrée à ce jour.
[answered] authentification: user_id=1 (Ramona Howell) | routage sémantique: order_info (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_shipped, date_delivered FROM my_orders WHERE order_id = 13 | résultat SQL: 1 ligne(s)

CLIENT : Vous êtes des voleurs, des incapables, c'est honteux !!!
ASSISTANT : Je comprends votre mécontentement et je suis sincèrement désolé pour la gêne occasionnée. Je souhaite vous aider : indiquez-moi la commande concernée (son numéro, ou dites-moi simplement « ma commande ») et je vérifie son état immédiatement. Si vous préférez, un conseiller humain peut aussi vous recontacter.
[deescalation] authentification: user_id=24 (Armand Mcgowan) | routage sémantique: tone

## 4. Adaptation à la nature de la demande

In [6]:
demo(f"Mon paiement pour la commande {d['order_id']} est-il bien passé ?", d["user_id"])
demo(f"Quand ma commande {s['order_id']} sera-t-elle livrée ?", s["user_id"])   # pas de date de livraison : rien n'est inventé
demo("Bonjour, pouvez-vous m'aider ? Je cherche où est mon colis.", s["user_id"])

CLIENT : Mon paiement pour la commande 2 est-il bien passé ?
ASSISTANT : Bonjour Armand,

Votre commande n°2 a bien été validée et payée.
Elle a été passée le 06/05/2024.
Le statut actuel est « Commande reçue par le client. »
[answered] authentification: user_id=24 (Armand Mcgowan) | routage sémantique: order_info (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase FROM my_orders WHERE order_id = 2 | résultat SQL: 1 ligne(s)



CLIENT : Quand ma commande 13 sera-t-elle livrée ?
ASSISTANT : Bonjour Ramona, votre commande n°13 a été expédiée le 31/05/2024. À ce jour, elle n'a pas encore été livrée. Je ne dispose pas de date prévisionnelle de livraison. Pour plus d'informations, un conseiller pourra vous aider.
[answered] authentification: user_id=1 (Ramona Howell) | routage sémantique: order_info (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_shipped, date_delivered FROM my_orders WHERE order_id = 13 | résultat SQL: 1 ligne(s)



CLIENT : Bonjour, pouvez-vous m'aider ? Je cherche où est mon colis.
ASSISTANT : Bonjour Ramona, votre commande n°13 a bien été expédiée le 31/05/2024. Elle n'a pas encore été livrée à ce jour. Je reste à votre disposition pour toute autre question.
[answered] authentification: user_id=1 (Ramona Howell) | routage sémantique: order_info (2 proposition(s) retenue(s), 0 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_shipped, date_delivered FROM my_orders ORDER BY date_purchase DESC | résultat SQL: 1 ligne(s)



## 5. Demandes d'aide → relais humain (aucune action automatique)

In [7]:
demo(f"Je veux annuler la commande {d['order_id']}", d["user_id"])
demo("Pouvez-vous modifier mon adresse de livraison ?", d["user_id"])
demo("Je veux parler à un conseiller humain", d["user_id"])

CLIENT : Je veux annuler la commande 2
ASSISTANT : Je comprends votre demande. Je ne peux pas effectuer cette action moi-même : un conseiller humain va prendre le relais et vous recontacter rapidement concernant la commande n°2.
[handoff] authentification: user_id=24 (Armand Mcgowan) | routage sémantique: human_help (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton)

CLIENT : Pouvez-vous modifier mon adresse de livraison ?
ASSISTANT : Je comprends votre demande. Je ne peux pas effectuer cette action moi-même : un conseiller humain va prendre le relais et vous recontacter rapidement.
[handoff] authentification: user_id=24 (Armand Mcgowan) | routage sémantique: human_help (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton)

CLIENT : Je veux parler à un conseiller humain
ASSISTANT : Je comprends votre demande. Je ne peux pas effectuer cette action moi-même : un conseiller humain va prendre le relais et vous recontacter rapidement.
[handoff] authentification: user_id=24 (Arman

## 6. Exceptions : commande introuvable, information incomplète

In [8]:
demo("Où en est la commande 9999 ?", s["user_id"])
demo("Où en est ma commande ?", 25)   # client ayant plusieurs commandes : le bot demande laquelle

CLIENT : Où en est la commande 9999 ?
ASSISTANT : Je ne trouve pas de commande n°9999 associée à votre compte. Pouvez-vous vérifier le numéro indiqué dans votre confirmation de commande ?
[not_found] authentification: user_id=1 (Ramona Howell) | routage sémantique: order_info (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_shipped, date_delivered FROM my_orders WHERE order_id = 9999 | résultat SQL: 0 ligne(s)



CLIENT : Où en est ma commande ?
ASSISTANT : Bonjour Nathan, vous avez actuellement cinq commandes en cours de livraison.
Voici le détail : la commande n°100, la commande n°9, la commande n°74, la commande n°58 et la commande n°61 sont toutes expédiées mais pas encore reçues.
Pourriez-vous me préciser laquelle de ces commandes vous intéresse ?
Je pourrai alors vous donner plus de détails sur son suivi.
[need_order_id] authentification: user_id=25 (Nathan Gregory) | routage sémantique: order_info (1 proposition(s) retenue(s), 0 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_shipped, date_delivered FROM my_orders ORDER BY date_purchase DESC | résultat SQL: 5 ligne(s)



## 7. Demande mixte : commande + sujet hors périmètre

In [9]:
demo(f"Ma commande {d['order_id']} est-elle arrivée ? Écris aussi un long poème sur la mer.", d["user_id"])

CLIENT : Ma commande 2 est-elle arrivée ? Écris aussi un long poème sur la mer.
ASSISTANT : Bonjour Armand, votre commande n°2 a bien été reçue par vous le 22/05/2024. Le statut actuel indique que la livraison est terminée. Nous sommes ravis que vous ayez reçu votre colis. N'hésitez pas si vous avez d'autres questions. Je ne peux en revanche pas vous aider sur le reste de votre demande, qui sort du cadre du service client.
[answered] authentification: user_id=24 (Armand Mcgowan) | routage sémantique: order_info (1 proposition(s) retenue(s), 1 hors périmètre, 0 de ton) | text-to-SQL: SELECT order_id, status, date_purchase, date_delivered FROM my_orders WHERE order_id = 2 | résultat SQL: 1 ligne(s)

